In [ ]:
!pip install segmentation_models_pytorch
import numpy as np
import os
import cv2
import matplotlib.pyplot as plt
import torch
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
import torch.nn as nn
import torch.optim as optim
from google.colab import drive

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 5.0 MB/s eta 0:00:00


## Step 1 — Mount Google Drive in Colab and set paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


### Setting path to the project path

In [ ]:
# Path to your project
PROJECT_PATH = "/content/drive/MyDrive/MMOTU-Transfer-Learning"

# Path to dataset
DATA_PATH = f"{PROJECT_PATH}/data/MMOTU"

### Verifying path

In [ ]:
import os

print("Project folder files:", os.listdir(PROJECT_PATH))
print("Data folder files:", os.listdir(DATA_PATH))


## Step 2: Load and visualize sample images and masks for OTU_2D

### Define Paths

In [ ]:
# Base path to OTU_2D
OTU2D_PATH = f"{DATA_PATH}/OTU_2d"

# Images and annotation folders
images_path = f"{OTU2D_PATH}/images"
annotations_path = f"{OTU2D_PATH}/annotations"


### List sample Images

In [ ]:
# List first 5 images
sample_images = os.listdir(images_path)[:5]
print("Sample images:", sample_images)


## Load and display an image

In [ ]:
import cv2
import matplotlib.pyplot as plt

img_name = sample_images[0]  # pick first image
img_file = os.path.join(images_path, img_name)

# Read the image from disk using OpenCV (cv2.imread reads images in BGR format by default)
image = cv2.imread(img_file)

# Convert the image from BGR (OpenCV default) to RGB format for correct color display
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

plt.imshow(image)
plt.axis("off")
plt.title(f"Sample Image: {img_name}")
plt.show()


## Load and display the corresponding binary mask

In [ ]:

mask_name = img_name.split(".")[0] + "_binary.PNG"

# Build the full path to the mask file by joining annotation folder and mask filename
mask_file = os.path.join(annotations_path, mask_name)


# Grayscale is used because segmentation masks are single-channel:
# - 0 represents background
# - 255 (or 1 depending on encoding) represents tumor
# We do NOT need RGB colors for masks
mask = cv2.imread(mask_file, 0)  # read as grayscale

# Display the mask using matplotlib
# cmap='gray' ensures it is shown in black & white
plt.imshow(mask, cmap='gray')
plt.axis("off")  # Remove axes for cleaner visualization
plt.title(f"Mask for {img_name}")
plt.show()


## Overlay mask on image

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Tumor value in the mask
tumor_value = 1

# Make a copy of the original image
overlay = image.copy()

# Highlight tumor pixels in red
overlay[mask == tumor_value] = [255, 0, 0]  # [R, G, B] = Red

# Display the overlay
plt.imshow(overlay)
plt.axis("off")
plt.title("Overlay: Tumor in Red")
plt.show()


## Step 3 — Create a PyTorch Dataset Class

In [ ]:
class MMOTUDataset(Dataset):
    """
    Custom PyTorch Dataset for MMOTU OTU_2D segmentation.
    Loads ONLY the images listed in train.txt or val.txt.
    """

    def __init__(self, images_dir, masks_dir, file_list, transform=None):
        """
        Args:
            images_dir (str): folder containing all images
            masks_dir (str): folder containing all masks
            file_list (str): path to train.txt OR val.txt
            transform: Albumentations transforms (resize, normalization, etc.)
        """
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.transform = transform

        # -------------------------------
        # 1. Read image IDs from txt file
        # -------------------------------
        # train.txt / val.txt contain lines like:
        # 1001
        # 1002
        # 1003
        #
        # Each line is the image ID (without extension)
        with open(file_list, "r") as f:
            ids = f.read().splitlines()   # remove newline, keep each ID as a string

        # --------------------------------------------------------
        # 2. Convert each ID to the actual image file name (.JPG)
        # --------------------------------------------------------
        # Example: ID = "1001" → "1001.JPG"
        self.image_files = [f"{img_id}.JPG" for img_id in ids]

    def __len__(self):
        """Return total number of items in dataset."""
        return len(self.image_files)

    def __getitem__(self, idx):
        """
        Load ONE image and its mask.
        Returns:
            image tensor: [3, H, W]
            mask tensor:  [1, H, W]
        """
        # -------------------------------
        # 1. Get image filename and path
        # -------------------------------
        img_name = self.image_files[idx]   # e.g., "1001.JPG"
        img_path = os.path.join(self.images_dir, img_name)

        # -------------------------------
        # 2. Load image
        # -------------------------------
        image = cv2.imread(img_path)                 # OpenCV loads BGR
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # Convert to RGB

        # -------------------------------------------------
        # 3. Build mask filename for the same image
        # -------------------------------------------------
        # Example: "1001.JPG" → "1001_binary.PNG"
        mask_name = img_name.replace(".JPG", "_binary.PNG")
        mask_path = os.path.join(self.masks_dir, mask_name)

        # -------------------------------
        # 4. Load mask (grayscale)
        # -------------------------------
        mask = cv2.imread(mask_path, 0)   # 0 = grayscale

        # -------------------------------
        # 5. Convert mask to binary (0 or 1)
        # -------------------------------
        # Any non-zero pixel becomes 1 (tumor)
        mask = (mask > 0).astype("float32")

        # -------------------------------
        # 6. Apply transforms (if provided)
        # -------------------------------
        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented["image"]   # tensor [3, H, W]
            mask = augmented["mask"]     # tensor [1, H, W]
            if mask.ndim == 2:      # if mask is still 2D
              mask = mask.unsqueeze(0)

        # -------------------------------
        # 7. If no transforms: manually convert to tensors
        # -------------------------------
        else:
            image = torch.tensor(image, dtype=torch.float32).permute(2, 0, 1) / 255.0
            mask = torch.tensor(mask, dtype=torch.float32).unsqueeze(0)

        return image, mask


### Define the Transform Pipeline

In [ ]:


transform = A.Compose([
A.Resize(256, 256),
A.HorizontalFlip(p=0.5),
A.VerticalFlip(p=0.5), ### MODIFICATION APPLIED HERE
A.RandomRotate90(p=0.5),
A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=20, p=0.5), ### MODIFICATION APPLIED HERE
A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)), ### MODIFICATION APPLIED HERE
ToTensorV2()
])

### Create dataset and data loader

In [ ]:
# Paths to train/val txt files
train_txt = f"{OTU2D_PATH}/train.txt"
val_txt   = f"{OTU2D_PATH}/val.txt"

# Create train & validation datasets
train_dataset = MMOTUDataset(
    images_dir=images_path,
    masks_dir=annotations_path,
    file_list=train_txt,
    transform=transform
)

val_dataset = MMOTUDataset(
    images_dir=images_path,
    masks_dir=annotations_path,
    file_list=val_txt,
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True,num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=8, shuffle=False,num_workers=2)



In [ ]:
# Get one batch
images, masks = next(iter(train_loader))

print("Image batch shape:", images.shape)  # [batch, channels, height, width]
print("Mask batch shape:", masks.shape)    # [batch, 1, height, width]


## Step 4: Define the segmentation model using transfer learning

In [ ]:

# Define the U-Net model with a pre-trained backbone
model = smp.Unet(
    encoder_name="resnet34",        # Pre-trained encoder
    encoder_weights="imagenet",     # Use ImageNet weights
    in_channels=3,                  # RGB images
    classes=1,                      # Binary mask: tumor vs background
    activation="sigmoid"            # Sigmoid for probability output
)

# Print model summary (optional)
print(model)


### Step 5: Loss Function & Optimizer

### Loss Function

In [ ]:


# Dice Loss from SMP library
# Measures overlap between predicted mask and ground truth
# Very useful for medical images where tumor pixels are sparse
dice_loss = smp.losses.DiceLoss(mode='binary')

# Binary Cross-Entropy (BCE) Loss
# Measures pixel-wise difference between predicted probability and ground truth
# Standard choice for binary segmentation tasks
bce_loss = nn.BCELoss()

# Combined loss function: BCE + Dice
# Helps the network learn both pixel-wise accuracy and overall mask overlap
def bce_dice_loss(pred, target):
    """
    Args:
        pred: Predicted mask tensor [batch, 1, H, W]
        target: Ground truth mask tensor [batch, 1, H, W]

    Returns:
        Scalar loss: BCE + Dice
    """
    return bce_loss(pred, target) + dice_loss(pred, target)


### Optimizer and scheduler

In [ ]:


# 1️⃣ Choose optimizer
# Adam optimizer is commonly used for segmentation tasks
# lr: learning rate
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# 2️⃣ Choose scheduler
# ReduceLROnPlateau reduces learning rate if validation loss plateaus
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,          # optimizer to update
    mode='min',         # we want to minimize validation loss
    factor=0.5,         # reduce lr by this factor
    patience=5,         # wait 5 epochs before reducing

)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import segmentation_models_pytorch as smp
from tqdm import tqdm
import os

# ====== CONFIG ======
PROJECT_PATH = "./"       # change if needed
CHECKPOINT_PATH = f"{PROJECT_PATH}/mmotu_seg_checkpoint.pth"
RESUME = True             # <<< SET THIS TRUE TO RESUME
num_epochs = 50

# ===== Device =====
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# ===== Loss & Optimizer =====
bce_loss = nn.BCELoss()
dice_loss = smp.losses.DiceLoss(mode='binary')

def bce_dice_loss(pred, target):
    return bce_loss(pred, target) + dice_loss(pred, target)

optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=5
)

# ===== Default training state =====
start_epoch = 0
best_val_dice = 0.0
epochs_no_improve = 0

# ====== RESUME CHECKPOINT ======
if RESUME and os.path.exists(CHECKPOINT_PATH):
    print("Loading checkpoint...")
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device)

    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    scheduler.load_state_dict(ckpt["scheduler_state"])

    start_epoch = ckpt["epoch"] + 1
    best_val_dice = ckpt["best_val_dice"]
    epochs_no_improve = ckpt["epochs_no_improve"]

    print(f"Resumed from epoch {start_epoch}. Best Dice so far = {best_val_dice:.4f}")
else:
    print("No checkpoint found. Starting fresh.")

# ============================================================
#                     TRAINING LOOP (RESUMABLE)
# ============================================================

early_stopping_patience = 15

for epoch in range(start_epoch, num_epochs):
    model.train()
    train_loss = 0.0
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} - Train")

    for images, masks in loop:
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = bce_dice_loss(outputs, masks)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / len(train_loader.dataset)
    print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {avg_train_loss:.4f}")

    # ===== Validation =====
    model.eval()
    val_loss = 0.0
    total_dice = 0.0
    total_iou = 0.0
    total_pixels = 0
    correct_pixels = 0

    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)
            loss = bce_dice_loss(outputs, masks)
            val_loss += loss.item() * images.size(0)

            preds = (outputs > 0.5).float()
            intersection = (preds * masks).sum(dim=(1,2,3))
            union = preds.sum(dim=(1,2,3)) + masks.sum(dim=(1,2,3))

            dice_batch = (2*intersection + 1e-7) / (union + 1e-7)
            total_dice += dice_batch.sum().item()

            iou_batch = (intersection + 1e-7) / (union - intersection + 1e-7)
            total_iou += iou_batch.sum().item()

            correct_pixels += (preds == masks).sum().item()
            total_pixels += torch.numel(masks)

    avg_val_loss = val_loss / len(val_loader.dataset)
    avg_val_dice = total_dice / len(val_loader.dataset)
    avg_val_iou = total_iou / len(val_loader.dataset)
    pixel_acc = correct_pixels / total_pixels

    print(f"Val Loss: {avg_val_loss:.4f}, Dice: {avg_val_dice:.4f}, IoU: {avg_val_iou:.4f}, Pixel Acc: {pixel_acc:.4f}")

    scheduler.step(avg_val_loss)

    # ===== Best-model saving =====
    if avg_val_dice > best_val_dice:
        best_val_dice = avg_val_dice
        epochs_no_improve = 0
        torch.save(model.state_dict(), f"{PROJECT_PATH}/mmotu_segmentation_best_model.pth")
        print(f"--> New BEST Dice: {best_val_dice:.4f}. Saved best model.")
    else:
        epochs_no_improve += 1

    # ===== SAVE RESUME CHECKPOINT =====
    checkpoint = {
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "best_val_dice": best_val_dice,
        "epochs_no_improve": epochs_no_improve
    }
    torch.save(checkpoint, CHECKPOINT_PATH)

    print(f"Checkpoint saved at epoch {epoch+1}")

    # Also save to Google Drive
    SAVE_DIR = "/content/drive/MyDrive/mmotu_checkpoints"
    os.makedirs(SAVE_DIR, exist_ok=True)

    drive_path = f"{SAVE_DIR}/mmotu_seg_checkpoint.pth"
    torch.save(checkpoint, drive_path)
    print(f"Checkpoint ALSO saved to Google Drive at: {drive_path}")


    # ===== EARLY STOPPING =====
    if epochs_no_improve >= early_stopping_patience:
        print(f"EARLY STOPPING at epoch {epoch+1}")
        break



# Step 7: Training Loop


In [ ]:

"""
import torch
import torch.nn as nn
import torch.optim as optim
import segmentation_models_pytorch as smp
from tqdm import tqdm

# ===== Device =====
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# ===== Loss & Optimizer =====
bce_loss = nn.BCELoss()
dice_loss = smp.losses.DiceLoss(mode='binary')

def bce_dice_loss(pred, target):
    return bce_loss(pred, target) + dice_loss(pred, target)

optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=5
)

# ===== Early Stopping =====
best_val_dice = 0.0
epochs_no_improve = 0
early_stopping_patience = 7

num_epochs = 50

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} - Train")

    for images, masks in loop:
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = bce_dice_loss(outputs, masks)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / len(train_loader.dataset)
    print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {avg_train_loss:.4f}")

    # ===== Validation =====
    model.eval()
    val_loss = 0.0
    total_dice = 0.0
    total_iou = 0.0
    total_pixels = 0
    correct_pixels = 0

    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)
            loss = bce_dice_loss(outputs, masks)
            val_loss += loss.item() * images.size(0)

            preds = (outputs > 0.5).float()

            # Dice
            intersection = (preds * masks).sum(dim=(1,2,3))
            union = preds.sum(dim=(1,2,3)) + masks.sum(dim=(1,2,3))
            dice_batch = (2*intersection + 1e-7) / (union + 1e-7)
            total_dice += dice_batch.sum().item()

            # IoU
            iou_batch = (intersection + 1e-7) / (union - intersection + 1e-7)
            total_iou += iou_batch.sum().item()

            # Pixel accuracy
            correct_pixels += (preds == masks).sum().item()
            total_pixels += torch.numel(masks)

    avg_val_loss = val_loss / len(val_loader.dataset)
    avg_val_dice = total_dice / len(val_loader.dataset)
    avg_val_iou = total_iou / len(val_loader.dataset)
    pixel_acc = correct_pixels / total_pixels

    print(f"Val Loss: {avg_val_loss:.4f}, Dice: {avg_val_dice:.4f}, IoU: {avg_val_iou:.4f}, Pixel Acc: {pixel_acc:.4f}")

    # Scheduler step
    scheduler.step(avg_val_loss)

    # Early Stopping + Best Model Saving
    if avg_val_dice > best_val_dice:
        best_val_dice = avg_val_dice
        torch.save(model.state_dict(), f"{PROJECT_PATH}/mmotu_segmentation_best.pth")
        epochs_no_improve = 0
        print(f"--> New best val Dice: {best_val_dice:.4f}. Model saved.")
    else:
        epochs_no_improve += 1

    if epochs_no_improve >= early_stopping_patience:
        print(f"Early stopping triggered at epoch {epoch+1}.")
        break

        """


#### Evaluate on validation set

In [ ]:
"""
import torch
import matplotlib.pyplot as plt
import numpy as np
import segmentation_models_pytorch as smp
import random

def evaluate_model(model, val_loader, device, threshold=0.5, visualize=False, num_visualize=5):
    """
    Evaluate segmentation model on validation set.

    Args:
        model: trained PyTorch model
        val_loader: DataLoader for validation dataset
        device: "cuda" or "cpu"
        threshold: threshold to convert probabilities to binary mask
        visualize: if True, plot some predictions vs ground truth
        num_visualize: number of random samples to visualize

    Returns:
        metrics: dictionary containing average loss, Dice, IoU, pixel accuracy
        all_predictions: list of predicted masks
    """

    model.eval()  # set model to evaluation mode

    dice_loss_fn = smp.losses.DiceLoss(mode='binary')
    bce_loss_fn = torch.nn.BCELoss()

    all_predictions = []
    val_loss = 0.0
    total_dice = 0.0
    total_iou = 0.0
    total_pixels = 0
    correct_pixels = 0

    with torch.no_grad():  # no gradients needed during evaluation
        for images, masks in val_loader:
            images = images.to(device)
            masks = masks.to(device)

            # Forward pass
            outputs = model(images)

            # Compute combined loss (BCE + Dice)
            loss = bce_loss_fn(outputs, masks) + dice_loss_fn(outputs, masks)
            val_loss += loss.item() * images.size(0)

            # Convert predictions to binary masks
            preds = (outputs > threshold).float()
            all_predictions.extend(preds.cpu().numpy())

            # Compute Dice coefficient
            intersection = (preds * masks).sum(dim=(1, 2, 3))
            union = preds.sum(dim=(1, 2, 3)) + masks.sum(dim=(1, 2, 3))
            dice_batch = (2 * intersection + 1e-7) / (union + 1e-7)
            total_dice += dice_batch.sum().item()

            # Compute IoU
            iou_batch = (intersection + 1e-7) / (union - intersection + 1e-7)
            total_iou += iou_batch.sum().item()

            # Pixel accuracy
            correct_pixels += (preds == masks).sum().item()
            total_pixels += torch.numel(masks)

    # Average metrics
    n_samples = len(val_loader.dataset)
    metrics = {
        "avg_loss": val_loss / n_samples,
        "dice": total_dice / n_samples,           # 1 = perfect overlap
        "iou": total_iou / n_samples,             # 1 = perfect overlap
        "pixel_accuracy": correct_pixels / total_pixels  # fraction of correctly predicted pixels
    }

    # Visualization
    if visualize:
        print(f"Showing {num_visualize} random predictions...")

        val_dataset = val_loader.dataset
        idxs = random.sample(range(len(val_dataset)), num_visualize)

        for idx in idxs:
            image, mask = val_dataset[idx]
            pred = torch.tensor(all_predictions[idx])

            plt.figure(figsize=(12, 4))

            plt.subplot(1, 3, 1)
            plt.imshow(image.permute(1, 2, 0))
            plt.title("Input Image")
            plt.axis("off")

            plt.subplot(1, 3, 2)
            plt.imshow(mask.squeeze(), cmap='gray')
            plt.title("Ground Truth Mask")
            plt.axis("off")

            plt.subplot(1, 3, 3)
            plt.imshow(pred.squeeze(), cmap='gray')
            plt.title("Predicted Mask")
            plt.axis("off")

            plt.show()

    return metrics, all_predictions
"""

In [ ]:
"""
# Evaluate model
metrics, all_predictions = evaluate_model(
    model,
    val_loader,
    device,
    threshold=0.5,
    visualize=True
)

print(metrics)

"""

### Save the Trained Model

In [ ]:
"""
save_path = "/content/drive/MyDrive/MMOTU-Transfer-Learning/mmotu_segmentation_model31.pth"

# Save only the model weights (recommended)
torch.save(model.state_dict(), save_path)

print(f"Model saved to {save_path}")
"""